In [ ]:
import torch
import numpy as np
import pandas as pd
import json
from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.6 GB


In [ ]:
# Load balanced master dataset
df = pd.read_csv('../data/processed/master_dataset_balanced.csv')

# Load label maps
with open('../data/processed/label2id.json') as f:
    label2id = json.load(f)
with open('../data/processed/id2label.json') as f:
    id2label = json.load(f)

# Convert id2label keys back to integers
id2label = {int(k): v for k, v in id2label.items()}

NUM_LABELS = len(label2id)
print(f"Total samples : {len(df)}")
print(f"Num classes   : {NUM_LABELS}")
print(f"\nLabel mapping :")
for label, idx in sorted(label2id.items()):
    count = (df['label'] == label).sum()
    print(f"  {idx:2d} → {label:<25} ({count} samples)")

In [ ]:
# Stratified split to preserve class distribution
# 80% train | 10% validation | 10% test
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print(f"Train samples      : {len(train_df)}")
print(f"Validation samples : {len(val_df)}")
print(f"Test samples       : {len(test_df)}")

# Verify class distribution is preserved
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts())

# Save test set separately — never touched during training
test_df.to_csv('../data/processed/test_set.csv', index=False)
print("\nTest set saved to test_set.csv")

In [ ]:
# Load BERT tokenizer
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128  # covers 95%+ of requirement sentences

def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'label_id']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[['text', 'label_id']].reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df[['text', 'label_id']].reset_index(drop=True))

# Rename label_id to labels (required by HuggingFace Trainer)
train_dataset = train_dataset.rename_column('label_id', 'labels')
val_dataset   = val_dataset.rename_column('label_id', 'labels')
test_dataset  = test_dataset.rename_column('label_id', 'labels')

# Tokenize
print("Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset   = val_dataset.map(tokenize_batch, batched=True)
test_dataset  = test_dataset.map(tokenize_batch, batched=True)

# Set format for PyTorch
cols = ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
train_dataset.set_format(type='torch', columns=cols)
val_dataset.set_format(type='torch', columns=cols)
test_dataset.set_format(type='torch', columns=cols)

print(f"Tokenization complete")
print(f"   Train : {len(train_dataset)} samples")
print(f"   Val   : {len(val_dataset)} samples")
print(f"   Test  : {len(test_dataset)} samples")

In [ ]:
# Load model
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

model = model.to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable:,}")
print(f"Model loaded to      : {device}")

In [ ]:
# Training configuration
# Settings are tuned for RTX 4060 Laptop (8.6GB VRAM)
BATCH_SIZE    = 16    # safe for 8.6GB — increase to 32 if no OOM error
EPOCHS        = 5
LEARNING_RATE = 2e-5  # standard BERT fine-tuning rate
WARMUP_RATIO  = 0.1   # 10% warmup steps

training_args = TrainingArguments(
    output_dir                  = '../models/bert_requirements',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = 0.01,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_dir                 = '../models/logs',
    logging_steps               = 50,
    fp16                        = True,   # RTX 4060 supports fp16 — halves VRAM usage
    dataloader_num_workers      = 2,
    report_to                   = 'none', # disable wandb
    save_total_limit            = 2,      # keep only best 2 checkpoints
    seed                        = 42,
)

print("Training configuration:")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Epochs        : {EPOCHS}")
print(f"  Learning rate : {LEARNING_RATE}")
print(f"  fp16          : True (VRAM optimized)")
print(f"  Warmup ratio  : {WARMUP_RATIO}")

In [ ]:
# Define metrics computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        'f1_macro'   : f1_score(labels, predictions, average='macro',    zero_division=0),
        'f1_weighted': f1_score(labels, predictions, average='weighted', zero_division=0),
        'precision'  : precision_score(labels, predictions, average='macro', zero_division=0),
        'recall'     : recall_score(labels, predictions, average='macro',    zero_division=0),
    }

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting training...")
print(f"Estimated time: 30–45 minutes on RTX 4060\n")

train_result = trainer.train()

print(f"\nTraining complete!")
print(f"   Total steps    : {train_result.global_step}")
print(f"   Training loss  : {train_result.training_loss:.4f}")
print(f"   Time taken     : {train_result.metrics['train_runtime']/60:.1f} minutes")

In [ ]:
# Evaluate on validation set
print("Evaluating on held-out test set...")
predictions = trainer.predict(test_dataset)
preds  = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Full classification report
label_names = [id2label[i] for i in range(NUM_LABELS)]
print("\n=== CLASSIFICATION REPORT ===\n")
print(classification_report(labels, preds, target_names=label_names, digits=4))

# Save metrics
test_f1 = f1_score(labels, preds, average='macro')
print(f"\nTest F1 (macro): {test_f1:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_names,
    yticklabels=label_names
)
plt.title('Confusion Matrix — BERT Requirements Classifier')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=150)
plt.show()
print("Saved confusion_matrix.png")

In [ ]:
# Save model + tokenizer
model_path = '../models/bert_requirements_final'
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

# Save label maps alongside the model
with open(f'{model_path}/label2id.json', 'w') as f:
    json.dump(label2id, f, indent=2)
with open(f'{model_path}/id2label.json', 'w') as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)

print(f"Model saved to {model_path}")
print(f"\nFiles saved:")
for f in os.listdir(model_path):
    size = os.path.getsize(f'{model_path}/{f}') / 1e6
    print(f"  {f:<40} {size:.1f} MB")